# Assignment 3 - Big Data Analytics using Apache Spark
## Student: Turgunboev Dadakhon
Description: Web server logs capture data about each registered event. They contain valuable information about site visitors, their behavior, the actions of search robots, business intelligence, and potential security threats.
* Dataset: Nginx server access log(3.26 GB) for 2019 year.
    * Columns: 'remote_addr', 'time_local', 'request', 'url', 'status', 'body_bytes_sent'
    * Sum of requests: 10269414   
    
* Assignment Tasks:
    * Exercises on Spark RDD
    * Exercises on Spark DataFrame
    * Exercises on Spark SQL
    * Exercises on Spark MLlib
* Prerequisites
    * Access to the university cluster

## Task 1: Exercises on Spark RDD
 1. Parses a single line of Nginx access log into structured fields.
 2. Create RDD with GET requests count per year
 3. Create an RDD that contains the unique ip addresses that has top 10 most successful GET requests.

In [ ]:
# Import the SparkSession class from PySpark SQL module
# This is the main entry point for DataFrame and SQL functionality
from pyspark.sql import SparkSession

# Create a SparkSession instance or get an existing one
# SparkSession is the unified entry point for Spark functionality
spark = SparkSession.builder.appName("Dadakhon-Assignment3").getOrCreate()

# SparkContext is the main connection to Spark cluster and represents the entry point for low-level APIs
sc = spark.sparkContext

# Set the log level to ERROR to reduce console output
# This suppresses INFO and WARN messages, showing only ERROR messages
sc.setLogLevel('ERROR')

# Print confirmation message
print("Spark session created successfully!")

In [ ]:
# Create an RDD (Resilient Distributed Dataset) by reading the log file
log_rdd = sc.textFile("data/dadakhon/access.log")
log_rdd.take(4)

### 1. Parses a single line of Nginx access log into structured fields.
* Args:
        line (str): A raw log line from Nginx access log
* Returns:
        dict: Parsed log fields as a dictionary, or None if parsing fails

In [ ]:
import re
from datetime import datetime

# Parse a line of nginx access log
def parse_log_line(line):
    pattern = (
        r'(?P<remote_addr>\S+) - - '                # Client IP address
        r'\[(?P<time_local>.*?)\] '                 # Timestamp in brackets
        r'"(?P<request_method>\S+) '                # HTTP method (GET/POST/etc)
        r'(?P<url>\S+) \S+" '                       # Request URL
        r'(?P<status>\d{3}) '                       # HTTP status code
        r'(?P<body_bytes_sent>\d+)'                # Response body size in bytes
    )

    # Attempt to match the pattern against the log line    
    match = re.search(pattern, line)
    if match:
        # Extract and parse the timestamp (ignoring timezone)
        time_str = match.group("time_local").split()[0]  
        dt = datetime.strptime(time_str, "%d/%b/%Y:%H:%M:%S")
        
        # Regular expression pattern to match Nginx log components
        return {
            "remote_addr": match.group("remote_addr"),
            "time_local": {
                "year": dt.year,
                "month": dt.month,
                "day": dt.day,
                "hours": dt.hour,
                "minutes": dt.minute,
                "seconds": dt.second
            },
            "request": match.group("request_method"),
            "url": match.group("url"),
            "status": int(match.group("status")),
            "body_bytes_sent": int(match.group("body_bytes_sent"))
        }
    else:
        # Return None for lines that don't match the pattern
        return None

In [ ]:
# Apply parser and filter out None values
parsed_rdd = log_rdd.map(parse_log_line).filter(lambda x: x is not None)

# Get the count of items in the parsed RDD
total_records = parsed_rdd.count()
print(f"Total number of log entries: {total_records}")

# Collect and display first 10 parsed records
parsed_rdd.take(10)

### 2. Create RDD with GET requests count per year
1. Show the first 10 results from the RDD sorted by most recent years. 